# Colab runner (thin)
Runtime -> Change runtime type -> **A100** (or T4). Secrets (key icon): `HF_TOKEN`, `GH_TOKEN` if private.
Outputs go to Google Drive (`MyDrive/mcp-poison-detect/outputs`) so they survive disconnects.

In [ ]:
REPO = "github.com/Arn15/mcp-poison-detect"
BRANCH = "main"
SCRIPT = "scripts/check_env.py"
ARGS = "--config configs/default.yaml"
GPU_DEPS = False

In [ ]:
import os, subprocess, base64
from google.colab import drive, userdata
drive.mount("/content/drive")
def secret(k):
    try: return userdata.get(k)
    except Exception: return None
HF_SECRET, GH_SECRET = "HF_TOKEN", "GH_TOKEN"     # names of your Colab secrets
os.environ["HF_TOKEN"] = secret(HF_SECRET) or ""
gh = secret(GH_SECRET)
WORK = "/content"

# Token goes in an HTTP header for this command only: never in the URL, .git/config, or error output.
auth = []
if gh:
    b64 = base64.b64encode(f"x-access-token:{gh}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {b64}"]
def git(*args, cwd=None):
    r = subprocess.run(["git", *auth, *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(gh or "\0", "***")
        raise RuntimeError(f"git {args[0]} failed: {msg.strip()}")
    return r.stdout

os.chdir(WORK)
repo_dir = os.path.join(WORK, "mcp-poison-detect")
if os.path.isdir(os.path.join(repo_dir, ".git")):
    git("fetch", "-q", cwd=repo_dir); git("checkout", "-q", BRANCH, cwd=repo_dir); git("pull", "-q", cwd=repo_dir)
else:
    git("clone", "-q", "-b", BRANCH, f"https://{REPO}.git", repo_dir)
os.chdir(repo_dir)
print(git("log", "-1", "--oneline", cwd=repo_dir))

In [ ]:
!pip install -q -r requirements.txt
if GPU_DEPS:
    !pip install -q -r requirements-gpu.txt
!pip install -q -e . --no-deps

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python {SCRIPT} {ARGS}